# One-Step Actor–Critic in a 4×4 Wumpus World (NumPy-only)This notebook trains a **One-Step Actor–Critic** agent (NOT A2C; one parameter update per environment step) on a **fully-observable** 4×4 Wumpus World, using only NumPy, matplotlib, and the Python standard library.Mechanism demonstrated:```State → Actor → Action → Environment → Reward + Next State → Critic → TD Target → TD Error → Actor/Critic Update```Goals:1. Hand-written forward/backward (no auto-diff, no PyTorch/TensorFlow/sklearn).2. Correct gradient signs for both Critic and Actor (verified by finite differences and direction tests).3. Vanilla SGD (no momentum).4. Clear train/test split: train on random maps (seeds 0–9999), test on held-out maps (seeds 10000–10199).5. Honest reporting of real results.

In [ ]:
import randomimport numpy as npimport matplotlib.pyplot as pltfrom actor_network import ActorNetworkfrom critic_network import CriticNetworkfrom nn import forward_with_cache, backward, sgd_step, softmax, samplefrom wumpus_env import WumpusWorldEnv, STATE_DIM, MAX_STEPS, ACTION_NAMESrandom.seed(0)np.random.seed(0)

## Backprop helpers (hand-written chain rule)- `forward_with_cache` stores each layer's `(input, pre-activation z, activation a)`.- `backward` computes `dW = outer(grad, input)`, `db = grad`, and propagates  `grad_to_previous = W.T @ grad`, applying the ReLU derivative `grad *= (z > 0)`.ReLU is `max(0, z)`; its derivative is `1` for `z>0`, `0` for `z<=0`.The final parameter update is **vanilla SGD**:$$W \leftarrow W - \eta \frac{\partial L}{\partial W}, \qquad b \leftarrow b - \eta \frac{\partial L}{\partial b}$$(no momentum, no velocity accumulation).

In [ ]:
# ----------------------------------------------------------------------# Hyperparameters (parameters are the network weights/biases, learned).# ----------------------------------------------------------------------LR_ACTOR = 1e-3      # learning rate for the actorLR_CRITIC = 1e-3     # learning rate for the criticGAMMA = 0.99         # discount factorENTROPY_COEF = 0.05  # entropy bonus coefficientSHAPE_KA = 1.0       # reward-shaping: pull toward goldSHAPE_KH = 0.5       # reward-shaping: push away from hazardsMAX_STEPS = 50EPISODES = 15000EVAL_EVERY = 500EVAL_EPISODES = 100TEST_SEED_BASE = 10000  # held-out test seeds start hereGRAD_CLIP = 20.0        # per-parameter gradient clipping (stability)def potential(env, ka=SHAPE_KA, kh=SHAPE_KH):    '''Potential function Phi(s) for potential-based reward shaping.'''    ax, ay = env.agent_pos    gx, gy = env.gold_pos    d_gold = abs(ax - gx) + abs(ay - gy)    hazards = env.pit_positions + [env.wumpus_pos]    d_hazard = min((abs(ax - hx) + abs(ay - hy)) for hx, hy in hazards)    return -ka * d_gold + kh * d_hazarddef sgd_step_clip(net, grads, lr, grad_clip):    '''Vanilla SGD with per-parameter gradient clipping (no momentum).'''    for i, (dW, db) in enumerate(grads):        W, b = net.layers[i]        dW = np.clip(dW, -grad_clip, grad_clip)        db = np.clip(db, -grad_clip, grad_clip)        W[:] = W - lr * dW        b[:] = b - lr * db        net.layers[i] = (W, b)

## Networks**Actor** `π_θ(a|s)`: `64 → 128 (ReLU) → 64 (ReLU) → 4` logits (no softmax in the last layer; softmax is applied for sampling).**Critic** `V_φ(s)`: `64 → 128 (ReLU) → 64 (ReLU) → 1` scalar output (no softmax).During training we **sample** an action from the categorical distribution (no argmax).After training we use `argmax` for the deterministic policy.

In [ ]:
actor = ActorNetwork(input_size=STATE_DIM, hidden_sizes=(128, 64), n_actions=4, seed=0)critic = CriticNetwork(input_size=STATE_DIM, hidden_sizes=(128, 64), seed=0)env = WumpusWorldEnv()print('Actor params:', [(W.shape, b.shape) for W, b in actor.layers])print('Critic params:', [(W.shape, b.shape) for W, b in critic.layers])

## Training loop (One-Step Actor–Critic)For each transition `(s, a, r, s', done)`:1. **TD Target** `y = r' + γ(1-done)·V_φ(s')`   (r' = shaped reward = `r + γΦ(s') − Φ(s)`)2. **TD Error** `δ = y − V_φ(s)`   (one-step estimator of the advantage `A(s,a) ≈ Q(s,a) − V(s)`)3. **Critic** loss `L_c = ½(y − V(s))²`; gradient `∂L_c/∂V = −δ`4. **Actor** loss `L_a = −log π(a|s)·δ`; gradient `∂L_a/∂logits = δ(p − onehot)`Terminal next value is fixed at `0` (`next_value = 0 if done else ...`).

In [ ]:
def train(episodes=EPISODES, verbose=True):    episode_rewards = []    success_flags = []    eval_returns, eval_success, eval_ep_numbers = [], [], []    for ep in range(episodes):        seed = ep % 10000  # training maps: seeds 0-9999        state = np.asarray(env.reset(seed=seed), dtype=np.float32)        done = False        total = 0.0        gold = False        pot_prev = potential(env)        while not done:            # Actor: forward + sample from categorical            logits, aacts = forward_with_cache(actor, state)            probs = softmax(logits)            action = sample(probs)            next_state_np, reward, done, _truncated, info = env.step(action)            total += reward            next_state = np.asarray(next_state_np, dtype=np.float32)            if info["outcome"] == "gold":                gold = True            # Reward shaping (potential-based; does not change optimal policy)            pot_next = 0.0 if done else potential(env)            shaped_reward = reward + (GAMMA * pot_next - pot_prev)            # TD target & TD error            value = critic.forward(state)            next_value = 0.0 if done else critic.forward(next_state)            target = shaped_reward + GAMMA * next_value            td_error = target - value            # Critic update: dL/dV = -delta            _, cacts = forward_with_cache(critic, state)            critic_grads = backward(critic, cacts, -np.array([td_error], np.float32))            sgd_step_clip(critic, critic_grads, LR_CRITIC, GRAD_CLIP)            # Actor update: dL/dlogits = delta*(p - onehot) - entropy term            onehot = np.zeros_like(probs)            onehot[action] = 1.0            d_logits = td_error * (probs - onehot)            entropy = -float(np.sum(probs * np.log(probs + 1e-8)))            d_logits = d_logits - ENTROPY_COEF * probs * (np.log(probs + 1e-8) + entropy)            d_logits = np.clip(d_logits, -5.0, 5.0).astype(np.float32)            actor_grads = backward(actor, aacts, d_logits)            sgd_step_clip(actor, actor_grads, LR_ACTOR, GRAD_CLIP)            state = next_state            pot_prev = pot_next if not done else pot_prev        episode_rewards.append(total)        success_flags.append(1.0 if gold else 0.0)        if (ep + 1) % EVAL_EVERY == 0:            sr, ar = evaluate(actor, env, EVAL_EPISODES)            eval_returns.append(ar)            eval_success.append(sr)            eval_ep_numbers.append(ep + 1)            if verbose:                print(f"Ep {ep+1:5d}/{episodes}  avg_r={np.mean(episode_rewards[-EVAL_EVERY:]):7.2f}  eval_sr={sr*100:5.1f}%")    return {        "episode_rewards": np.array(episode_rewards),        "eval_success": np.array(eval_success),        "eval_ep_numbers": np.array(eval_ep_numbers),    }def evaluate(actor, env, n_episodes=200):    '''Deterministic (argmax) evaluation on held-out test maps (seeds 10000+).'''    returns, n_gold = [], 0    for i in range(n_episodes):        seed = TEST_SEED_BASE + i        state = np.asarray(env.reset(seed=seed), dtype=np.float32)        done, total, steps = False, 0.0, 0        while not done and steps < MAX_STEPS:            action = int(np.argmax(actor.forward(state)))            state_np, reward, done, _trunc, info = env.step(action)            state = np.asarray(state_np, dtype=np.float32)            total += reward; steps += 1            if info["outcome"] == "gold":                n_gold += 1        returns.append(total)    return n_gold / n_episodes, float(np.mean(returns))res = train(episodes=EPISODES)

## Plot 1: Learning curve (episode return + moving average)

In [ ]:
episode_rewards = res["episode_rewards"]window = 100ma = np.convolve(episode_rewards, np.ones(window)/window, mode="valid")plt.figure(figsize=(10, 4))plt.plot(episode_rewards, alpha=0.3, label="episode return")plt.plot(ma, color="red", label=f"moving avg (window={window})")plt.axhline(0, color="gray", ls="--", lw=0.8)plt.xlabel("Episode"); plt.ylabel("Episode return")plt.title("Plot 1: One-Step Actor-Critic Learning Curve")plt.legend(); plt.grid(True, alpha=0.3)plt.tight_layout(); plt.show()

## Plot 2: Success rate during training (on held-out maps)

In [ ]:
plt.figure(figsize=(8, 4))plt.plot(res["eval_ep_numbers"], np.array(res["eval_success"])*100, marker="o", ms=4)plt.xlabel("Training episode")plt.ylabel("Success rate on held-out maps (%)")plt.title("Plot 2: Success Rate During Training")plt.grid(True, alpha=0.3)plt.tight_layout(); plt.show()

## Final test performance on unseen mapsThe agent is evaluated on test seeds `10000–10199` which were **never** used in training.

In [ ]:
from collections import Counterenv2 = WumpusWorldEnv()counts = Counter()all_ret, all_steps, gold_steps = [], [], []for i in range(200):    seed = TEST_SEED_BASE + i    state = np.asarray(env2.reset(seed=seed), dtype=np.float32)    done, total, steps = False, 0.0, 0    out = "move"    while not done and steps < MAX_STEPS:        action = int(np.argmax(actor.forward(state)))        state_np, reward, done, _trunc, info = env2.step(action)        state = np.asarray(state_np, dtype=np.float32)        total += reward; steps += 1        out = info["outcome"]    counts[out] += 1    all_ret.append(total); all_steps.append(steps)    if out == "gold":        gold_steps.append(steps)n = 200print("Outcomes over 200 unseen test maps:", dict(counts))print(f"Success rate : {counts['gold']/n*100:.1f}%")print(f"Hazard rate  : {(counts['pit']+counts['wumpus'])/n*100:.1f}% "      f"(pit {counts['pit']/n*100:.1f}%, wumpus {counts['wumpus']/n*100:.1f}%)")print(f"Timeout rate : {counts['max_steps']/n*100:.1f}%")print(f"Avg return   : {np.mean(all_ret):.2f}")print(f"Avg steps on successes : {np.mean(gold_steps):.2f}")